In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**0.1 ライブラリ読み込み**

In [ ]:
import pandas as pd  # データ解析ライブラリ
import numpy as np  # 数値計算ライブラリ
import matplotlib.pyplot as plt  # グラフ描画ライブラリ


**0.2 データセット読み込み**

In [ ]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
sample_submission = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')

train_images = train_df.drop('label', axis=1).values
train_labels = train_df['label'].values
test_images = test_df


**1. データ確認**

In [ ]:
print(train_images.shape)

# -> データ数：　４２０００枚, 画像サイズ: 28*28ピクセル -> 784

In [ ]:
# 画像を1枚だけグラフ表示
data_2d = train_images[1].reshape(28, 28)

# 表示
plt.imshow(data_2d, cmap='gray')
plt.colorbar()
plt.title("28x28 Reshaped Data")
plt.show()
# -> 各ピクセルに 0〜255の値が格納


In [ ]:
train_images[1]
# 出力結果は ”右クリック > Clear Cell Output"で消すことができる。

In [ ]:
print(train_labels)
# -> 各画像のラベル（どの数字かを示すアノテーション結果）が格納されている。


**1.1 データの前処理**

In [ ]:
# images の　0〜255は範囲が大きすぎるため、0〜1に変換
train_images = train_images / 255.0
test_images = test_images / 255.0

In [ ]:
# ===== 特徴量削減：定数列のdrop（＋任意で低分散列のdrop） =====
# まずは学習データの「ピクセル列」だけを取り出して列統計を計算
train_pixels_df = train_df.drop('label', axis=1)

# (1) 各列の最大・最小を計算し、max==min（定数列）を落とす
col_max = train_pixels_df.max(axis=0)
col_min = train_pixels_df.min(axis=0)
non_constant_mask = (col_max != col_min)

n_constant = (~non_constant_mask).sum()
print(f"[Constant cols] drop: {n_constant} / {train_pixels_df.shape[1]}")

# すでに正規化済みの train_images (ndarray) と test_images (DataFrame) に同じ列選択を適用
# ndarrayのtrain_imagesに列名を付けるため一旦DataFrame化
train_images_df = pd.DataFrame(train_images, columns=train_pixels_df.columns).loc[:, non_constant_mask]
test_images_df  = test_images.loc[:, non_constant_mask]  # test_imagesはこの時点でDataFrame（かつ0-1正規化済み）

# (2) 任意：低分散列のdrop（しない場合は var_thresh=None に）
var_thresh = None  # 例：1e-5 や 5e-5 などを設定すると低分散列をさらに除去
if var_thresh is not None:
    variances = train_images_df.var(axis=0)
    high_var_mask = (variances > var_thresh)
    n_lowvar = (~high_var_mask).sum()
    print(f"[Low-variance cols] drop: {n_lowvar} (threshold={var_thresh})")
    train_images_df = train_images_df.loc[:, high_var_mask]
    test_images_df  = test_images_df.loc[:, high_var_mask]

# 最終的にモデル学習用のndarrayへ戻す
train_images = train_images_df.values
test_images  = test_images_df.values

print(f"Final shape -> train_images: {train_images.shape}, test_images: {test_images.shape}")

In [ ]:
# 確認
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i].reshape(28, 28), cmap=plt.cm.binary) # reshape: 28*28の2次元に変換
    plt.xlabel(train_labels[i])
plt.show()

# -> 白黒が反転

**2.0 ライブラリ追加**

In [ ]:
from sklearn.tree import DecisionTreeClassifier  # 　sklearn : 機械学習ライブラリ
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

**2.1 さらに前処理**

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_images)
X_test = scaler.transform(test_images)
y_train = train_labels

# 平均：0, 分散:1に標準化（SVMの精度安定のため）
# Warning（カラム不一致）が出るが大丈夫

In [ ]:
# ===== モデル定義（random forest） =====
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)  # 決定木100本

# ===== 学習 =====
rf_model.fit(X_train, y_train)

# ===== 予測 =====
preds = rf_model.predict(X_test).astype(int)



In [ ]:
# ===== 提出ファイル作成 =====
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds)+1),
    "Label": preds
})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()



##### Accuracy：　0.96557